# Titanic — Exploratory Data Analysis

**Competition:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/c/titanic)
**Metric:** Accuracy
**Notebook:** 01 — EDA

---

### What this notebook does
Before touching any model, we need to deeply understand the data. This notebook follows a strict EDA checklist:
1. Load and inspect raw structure (shape, dtypes, nulls)
2. Understand the target variable (class balance)
3. Explore each feature's distribution
4. Understand relationships between features and survival

**Why this matters:** Every modelling decision downstream — what to impute, what to encode, what features to engineer — comes from what we find here. Rushing past EDA is the single most common cause of bad models.

## 1. Imports and Constants

We fix the random seed and set a consistent plot style at the very top. This makes results reproducible across every run.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Fix random seed for reproducibility across all operations that use randomness
SEED = 42
np.random.seed(SEED)

# Consistent plot style across all notebooks in this portfolio
sns.set_theme(style="whitegrid")

# Paths — relative to the competitions/titanic/ folder
TRAIN_PATH = "../data/train.csv"
TEST_PATH  = "../data/test.csv"

print("Libraries loaded.")

## 2. Load Data

We load both train and test together for EDA so we can spot structural differences (e.g. a feature that has nulls only in test). 

**Important:** We never fit any statistics (means, medians, encoders) on test — we only look at it here for structural awareness.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f"Train shape: {train.shape}")   # (rows, cols)
print(f"Test shape:  {test.shape}")
train.head()

## 3. Data Types, Nulls, and Basic Stats

**What to look for:**
- Columns with nulls — these need a decision (drop, impute, or flag)
- Data types — are numeric columns actually numeric, or stored as strings?
- Descriptive stats — any suspicious min/max values or unexpected ranges?

In [ ]:
print("=== Data Types ===")
print(train.dtypes)

print("\n=== Missing Values (Train) ===")
# Show count and percentage of nulls per column
null_counts = train.isnull().sum()
null_pct    = (null_counts / len(train) * 100).round(1)
missing_df  = pd.DataFrame({"count": null_counts, "pct": null_pct})
print(missing_df[missing_df["count"] > 0])

print("\n=== Missing Values (Test) ===")
null_counts_test = test.isnull().sum()
null_pct_test    = (null_counts_test / len(test) * 100).round(1)
missing_test_df  = pd.DataFrame({"count": null_counts_test, "pct": null_pct_test})
print(missing_test_df[missing_test_df["count"] > 0])

In [ ]:
train.describe()

## 4. Target Variable — Survival Distribution

Before anything else, understand what we're predicting and how balanced the classes are.

A heavily imbalanced target (e.g. 95% one class) would change our metric choice and modelling approach. With Titanic, we expect roughly 60/40 — worth confirming.

In [ ]:
print("Survival value counts:")
print(train["Survived"].value_counts())
print("\nSurvival rate (%):")
print((train["Survived"].value_counts(normalize=True) * 100).round(1))

# Bar chart of survival counts
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: raw counts
train["Survived"].value_counts().plot(kind="bar", ax=axes[0], color=["#d9534f", "#5cb85c"])
axes[0].set_title("Survival Counts")
axes[0].set_xticklabels(["Did not survive", "Survived"], rotation=0)
axes[0].set_ylabel("Count")

# Right: proportions — this is what the model is trying to predict
train["Survived"].value_counts(normalize=True).plot(kind="bar", ax=axes[1], color=["#d9534f", "#5cb85c"])
axes[1].set_title("Survival Rate")
axes[1].set_xticklabels(["Did not survive", "Survived"], rotation=0)
axes[1].set_ylabel("Proportion")

plt.tight_layout()
plt.show()

## 5. Survival Rate by Key Categorical Features

Now let's look at the features we hypothesised as important: **Sex**, **Pclass**, and **Embarked**.

For each one we plot survival rate — not just counts — because counts can mislead if the groups have very different sizes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

categorical_features = ["Sex", "Pclass", "Embarked"]

for ax, feature in zip(axes, categorical_features):
    # Group by feature and compute survival rate
    survival_rate = train.groupby(feature)["Survived"].mean().sort_values(ascending=False)
    survival_rate.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Survival Rate by {feature}")
    ax.set_ylabel("Survival Rate")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=0)
    # Add value labels on top of each bar
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=10)

plt.suptitle("Survival Rate by Categorical Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Numerical Feature Distributions by Survival

For numerical features like **Age** and **Fare**, we use KDE plots (smoothed histograms) split by survival status.

**What to look for:** If the distributions for survivors and non-survivors look different (different peaks, different spreads), that feature is likely useful to the model. If they overlap completely, the feature may not add much.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

numerical_features = ["Age", "Fare"]

for ax, feature in zip(axes, numerical_features):
    # Plot one KDE curve per survival class
    for survived_val, label, color in [(0, "Did not survive", "#d9534f"), (1, "Survived", "#5cb85c")]:
        subset = train[train["Survived"] == survived_val][feature].dropna()
        subset.plot(kind="kde", ax=ax, label=label, color=color, linewidth=2)
    ax.set_title(f"Distribution of {feature} by Survival")
    ax.set_xlabel(feature)
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

A correlation matrix shows pairwise linear relationships between all numerical features.

**What to look for:**
- Features that correlate strongly with `Survived` (good predictors)
- Features that correlate strongly with each other (multicollinearity — may cause redundancy)

Note: Correlation only captures *linear* relationships. A feature could be important even with low correlation if the relationship is non-linear.

In [ ]:
# Select only numeric columns for correlation
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

# Compute correlation matrix
corr_matrix = train[numeric_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,        # Show correlation values in each cell
    fmt=".2f",
    cmap="coolwarm",   # Red = positive correlation, Blue = negative
    center=0,
    square=True,
    linewidths=0.5
)
plt.title("Correlation Matrix — Numerical Features")
plt.tight_layout()
plt.show()

# Specifically print correlations with Survived, sorted
print("\nCorrelations with Survived:")
print(corr_matrix["Survived"].drop("Survived").sort_values(key=abs, ascending=False))

## 8. EDA Summary — Decisions for Next Notebook

After running this notebook, update `approach.md` with your findings. Here is the framework:

**Missing data decisions:**
- `Age` (~20% missing) → impute with **median grouped by Pclass and Sex** (more precise than global median)
- `Cabin` (~77% missing) → extract **deck letter** (first character) if present, else "Unknown"
- `Embarked` (2 missing) → fill with **mode** ("S")
- `Fare` (1 missing in test) → fill with **median**

**Columns to drop:**
- `PassengerId` — just a row ID, no predictive value (also leakage risk as an identifier)
- `Ticket` — high cardinality string, hard to use without deep feature engineering
- `Name` — raw name will be dropped, but **Title** (Mr, Mrs, Miss, Master) will be extracted from it

**Features to engineer in notebook 02:**
- `Title` — extracted from Name
- `FamilySize` = SibSp + Parch + 1
- `IsAlone` = 1 if FamilySize == 1, else 0
- `Deck` — from first character of Cabin

📝 **Reminder:** Update `../approach.md` with your findings and `../learnings.md` with anything that surprised you.